# NOTEBOOK FOR OBP CHAPTER

See word document `OBP_thesis_chapter` for write up.

Right, I started off with BAM files that had previously been mapped by Luke using `BWA mem` and `BWA aln` respectively (published in [Silver et al. (2025)](https://doi.org/10.1111/mec.17746) ).


```mermaid
flowchart TD

A[Raw reads] --> B[Mapping]
B --> Z[Dataset biases]
Z --> C[Variant calling tests]
C --> D[GL-based analyses]
C --> E[Genotype-based analyses]
D --> H[Filtering tests]
E --> H
H --> G[Temporal comparison]
```


## 3. BAM-LEVEL QC


### A. Quality control / Summary statistics

Check spreadsheet "BAM stats" for following stats: 

* mapping rate;
* duplicate rate;
* endogenous DNA content;
* mean and breadth of coverage;
* mapping-quality and base-quality distributions;
* fragment-length distributions;
* DNA-damage profiles for historical samples;
* contamination checks where feasible;
* sex inference;
* relatedness;
* sample identity and duplicate checks;
* mitochondrial coverage or haplotypes as an additional identity check;
* batch effects associated with library, sequencing run, historical/modern status or extraction protocol.

<details>
<summary> #####  SIDE TEST - REMOVE LOW QUALITY READS? (13/07/26) </summary>
I was debating wether to remove "low-quality" reads from the BAM files using `samtools view -q 20` but that filter was very harsh and was removing up to 30% reads. Additionally, I think using that filter is too harsh because historical genomes tend to have stronger reference bias and can map less sucesfully, so filtering by mapping quality can be harsh for historical DNA.\
I also considered using samtools flags like `-F 3844` to remove low quality alignments (e.g. unmapped reads and reads with unmapped mates, secondary alignments, QC-failed reads, duplicate reads) but those only seemed to remove <1% of reads so I considered it unnecessary. In conclusion, I decided it was best to keep the BAM files as they were and to filter later.
</details>


#### AMBER - PLOTTING MAPPING BIAS

There is a software called "AMBER" (https://github.com/tvandervalk/AMBER) which can plot different summary statistics. *Note: It can run max 6 BAM files at a time.*

To use it 


**Script:** [`amber_mapping_bias.sh`](OBP_chapter_scripts/amber_mapping_bias.sh)


#### QUALIMAP 2 


**Script:** [`qualimap.sh`](OBP_chapter_scripts/qualimap.sh)


#### Reference genome summary metrics


**Script:** [`ref_genome_metrics.sh`](OBP_chapter_scripts/ref_genome_metrics.sh)


### B. Sex identification

Silver et al. (2025) determined scaffold 5 as the sex chromosome. We did a test to identiy if there were other sex-linked scaffolds.

1. Calculate mean depth for every scaffold in every sample.
2. Normalise each scaffold by that sample’s autosomal depth.
3. Use scaffold_5 as the provisional Z chromosome to classify individuals.
4. Compare the inferred males and females across every scaffold.
5. Identify additional Z- and W-linked scaffolds.


**Script:** [`sex_scaffolds.sh`](OBP_chapter_scripts/sex_scaffolds.sh)


### C. Relatedness

I ran NGSrelate three times to avoid temporal structural biases: 1) All samples (N=35) 2) Modern samples only (N=19) 3) Historical only (N=16).\
Then I checked the kinship coefficients in R


**Script:** [`ngsrelate_combined_35.sh`](OBP_chapter_scripts/ngsrelate_combined_35.sh)


## 4. Historical DNA biases


### A. MapDamage 

**Script:** [`MapDamage.sh`](OBP_chapter_scripts/MapDamage.sh)


### B. Reference bias mapping 

**Script:** [`Reference_bias.sh`](OBP_chapter_scripts/Reference_bias.sh)


### C. HEARTY analysis

Study heterozygosity patterns 


### D. Dowsampled data 

We also downsampled data to 6x to see if the behaviour of the samples was very different - see [`Downsampling notebook`](../OBP_downsampling/OBP_downsampling.ipynb))


## 7. VARIANT GENERATION

I determined variants using two metods: ANGSD and snpAD. The first is a genotype likelihood-based approach designed to work with low coverage data. snpAD is a variant callerdesigned for ancient DNA that creates indivudal DNA damage and error models. I am testing both methods because the data is medium coverage and I could techincally use both methods.

> Note: I also did this pipeline with BAM files downsampled to 6x (using only indivudals that had an original mean coverage of >= 7x). See the Downsampling markdown for the pipeline used


### 7.1 ANGSD 
I am using two methodologies: 1) Luke's method (the researcher that generated this dataset and did a prelim analysis in a previous paper) 2) Methods that I am testing


#### 7.1.A. Luke's method

This script replicates the method to calculate heterozygosity using Lukes methods. I get pracitically the same heterozygosity values when I run this script, which I suspect are overinflated.


**Script:** [`Het_Luke.sh`](OBP_chapter_scripts/Het_Luke.sh)


##### Quality check


**Script:** [`het_summary_qc.sh`](OBP_chapter_scripts/het_summary_qc.sh)


#### 7.1.B Standard ANGSD method - INDIVIDUAL

Then, I wrote what I thought would be the best approach to reduce group biases and coverage biases when running ANGSD.

It differs to Lukes script by:

- Only uses scaffolds >100kb
- individual max depth rather than a general cap at 50X to avoid low complexity sites (i.e. repetitive)

I based this script on Fontsere et al. 2025 (Whooping Crane paper).


**Script:** [`Het_ANGSD.sh`](OBP_chapter_scripts/Het_ANGSD.sh)


> Results - ANGSD My method
>
> I have compared Lukes method and mine and we get similar values. Of course, there are difference: 1) I only used scaffolds >100kb 2) I didn't create a common group list first.
>
> The results are quite similar, although Lukes are slightly


#### 7.1.C Standard ANGSD method - GLOBAL COMMON SITES 

I am using the same parameters as the previous script but creating a common list of sites first with >80% data present and then calculating indivudal heterozygosity:


**Script:** [`Het_ANGSD_sharedlist80_global.sh`](OBP_chapter_scripts/Het_ANGSD_sharedlist80_global.sh)


#### 7.1.D Standard ANGSD method - PER GROUP COMMON SITES 

I am using the same parameters as the previous script but creating a common list of sites first with >80% data PER GROUP and then intersecting the sites and then calculating indivudal heterozygosity:


**Script:** [`Het_ANGSD_sharedlist80_pergroup.sh`](OBP_chapter_scripts/Het_ANGSD_sharedlist80_pergroup.sh)


I am also running a test where I want to compare the het if I use a snp p val filter:

**Script:** [`Het_ANGSD_sharedlist80_pergroup_testpval.sh`](OBP_chapter_scripts/Het_ANGSD_sharedlist80_pergroup_testpval.sh)




### 7.2 Genotype calling with snpAD


#### A. Variant call using snpAD

Because this is a pipeline it can be useful to have a script that will submit jobs and daisy chain them:


**Script:** [`submit_snpAD_pipeline_chain.sh`](OBP_chapter_scripts/submit_snpAD_pipeline_chain.sh)


And then to submit:


**Script:** [`make_snpAD_ToRun_chunks.sh`](OBP_chapter_scripts/make_snpAD_ToRun_chunks.sh)

**Script:** [`1_snpad.sh`](OBP_chapter_scripts/1_snpad.sh)



#### B. Filtering gVCF 

Basic filter removing outlier low and high depth sites for each indivudal  (based on 1% and 99% depth percentiles based on indivudal BAMs to establish maximum depth filter per sample).

Also removed low quality sites.

Create that script that we want to run per individual and scaffold:


**Script:** [`2_Filter.sh`](OBP_chapter_scripts/2_Filter.sh)


To create multiple SLURM scripts (1 per indivudal and scaffold) of the previous script: 


**Script:** [`submit_2_Filter_chunks.sh`](OBP_chapter_scripts/submit_2_Filter_chunks.sh)


After filtering, we calculate summary stats again.


#### C. MERGING INDIVIDUALS PER CHROM

To submit, simply write `sbatch --array=1-nlength <script>`


**Script:** [`3_Merge.sh`](OBP_chapter_scripts/3_Merge.sh)


Then run the summary statistics script. 


#### D. STRICTER FILTERING  - MISSINGNESS (80% vs 100%)

It is best to apply a per-group filter s rather than a global 80% filter.


**Script:** [`4_Missingness_filter.sh`](OBP_chapter_scripts/4_Missingness_filter.sh)


After, to check the missingness we can use
[`Missingness_check.sh`](OBP_chapter_scripts/Missingness_check.sh)


##### OLD - Global 80% filter


**Script:** [`4_Missingness_filter_OLD_global80.sh`](OBP_chapter_scripts/4_Missingness_filter_OLD_global80.sh)


> To assess the missingness of each indivudal in the miss80 script: [`missingness_check.sh`](OBP_chapter_scripts/Missingness_check.sh)



#### F. Extract bisnps


**Script:** [`5_bisnps_nosingletons.sh`](OBP_chapter_scripts/5_bisnps_nosingletons.sh)


#### G. Concat chromosomes 


**Script:** [`6_concat.sh`](OBP_chapter_scripts/6_concat.sh)


#### Quality check

See R script to plot summary statistics 


**Script:** [`gt_depth_qc_plots.R`](OBP_chapter_scripts/gt_depth_qc_plots.R)


Per indivudal, calculate the number of Hom Ref, Het and Hom Alt geontypes at each depth. Then calculate the total sites per depth and the ratio of Hom Ref, Het and Hom Alt per depth. After this, plot the change in ratio at each depth of Hom Ref, Het and Hom Alt in order to see the effect of depth. Do this for each individual so that you can see if there are any outlier inidvudals.


**Script:** [`GT_depth_QC.sh`](OBP_chapter_scripts/GT_depth_QC.sh)


Another quality checkis to test the mean depth and missingess of each scaffold:


**Script:** [`gt_depth_query.sh`](OBP_chapter_scripts/gt_depth_query.sh)



#### Summary stats

After running the pipeline, we can extract the summary stats from `bcftools stats`. 

**Script:** [`collect_all_stats.sh`](OBP_chapter_scripts/collect_all_stats.sh)


Calculating depth percentiles from VCFs and BAMs


**Script:** [`bam_depth_pct.sh`](OBP_chapter_scripts/bam_depth_pct.sh)